# GTSAM Double-Difference RTK with integer ambiguity resolution

This notebook walks through **gtsam-first RTK** step by step: cssrlib provides the
GNSS observation front-end, GTSAM's `DoubleDifference{Pseudorange,CarrierPhase}Factor`
(from the inuex35/gtsam fork) form the rover-base double differences and estimate a
static rover position + float ambiguities with **incremental ISAM2**, and the integers
are resolved with cssrlib's LAMBDA (`resamb_lambda`) through the shared
`gnss_ar` bridge.

Pipeline: **load RINEX -> per-epoch DD measurements -> reference satellite ->
build factor graph (ISAM2) -> integer AR -> evaluate**. Runs on the data bundled
with cssrlib (`src/cssrlib/data`).

In [1]:
import os
import numpy as np

import cssrlib.rinex as rn
import cssrlib.gnss as gn
from cssrlib.gnss import rSigRnx, uTYP, sat2prn
from cssrlib.rtk import rtkpos
import gtsam
from gtsam import symbol

from gnss_ar import resolve_ar

SYSS = (gn.uGNSS.GPS, gn.uGNSS.GAL)        # constellations to use
X = symbol('x', 0)                         # static rover ECEF position node
def AM(sat, f): return symbol('n', int(sat) * 10 + f)  # SD ambiguity node

## 1. Load RINEX (rover, base, navigation)

`rtkpos` is the cssrlib double-difference engine. We seed `nav.x[0:3]` with the
approximate rover position so its `qcedit` can compute satellite elevations.
`xyz_ref` is the surveyed marker used only to score accuracy.

In [2]:
bdir = os.path.join(os.path.dirname(os.getcwd()), 'src', 'cssrlib', 'data') + os.sep
xyz_ref = np.array([-3962108.673, 3381309.574, 3668678.638])
pos_ref = gn.ecef2pos(xyz_ref)

sigs = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("GL1C"), rSigRnx("GL2W"),
        rSigRnx("GS1C"), rSigRnx("GS2W"),
        rSigRnx("EC1C"), rSigRnx("EC5Q"), rSigRnx("EL1C"), rSigRnx("EL5Q"),
        rSigRnx("ES1C"), rSigRnx("ES5Q")]
sigsb = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("GL1C"), rSigRnx("GL2W"),
         rSigRnx("GS1C"), rSigRnx("GS2W"),
         rSigRnx("EC1X"), rSigRnx("EC5X"), rSigRnx("EL1X"), rSigRnx("EL5X"),
         rSigRnx("ES1X"), rSigRnx("ES5X")]

dec = rn.rnxdec(); dec.setSignals(sigs)
nav = gn.Nav(); dec.decode_nav(bdir + 'SEPT078M.21P', nav)
decb = rn.rnxdec(); decb.setSignals(sigsb)
decb.decode_obsh(bdir + '3034078M1.21O'); dec.decode_obsh(bdir + 'SEPT078M1.21O')
nav.rb = [-3959400.631, 3385704.533, 3667523.111]    # base station ECEF
rb = np.array(nav.rb)
rtk = rtkpos(nav, dec.pos)
nav.x[0:3] = np.array(dec.pos)                        # seed for elevations
nf = nav.nf
print('frequencies per constellation:', nf, ' base-rover baseline:',
      round(np.linalg.norm(xyz_ref - rb) / 1e3, 2), 'km')

frequencies per constellation: 2  base-rover baseline: 5.29 km


## 2. Front-end: per-epoch double-difference measurements

`prepare_double_difference_measurements` returns, per epoch, the rover/base
satellite positions (`rs`/`rsb`), the common-satellite indices (`iu`/`ir`),
the common satellites (`sat`) and rover elevations (`el`). No EKF -- this is just
the geometry/observation bundle that the GTSAM factors consume.

In [3]:
frames = []
sync = rn.sync_obs_hold(dec, decb, maxage=nav.maxtdiff)
for ne, (obs, obsb, dt) in enumerate(sync):
    if ne >= 60:
        break
    if obsb is None:
        continue
    dd = rtk.prepare_double_difference_measurements(obs, obsb, pos_pred=dec.pos)
    if dd is not None:
        frames.append((obs, obsb, dd))

obs0, obsb0, dd0 = frames[0]
print(f'{len(frames)} epochs collected')
print('epoch 0 common sats:', [int(s) for s in dd0.sat])
print('epoch 0 elevations [deg]:', np.round(np.rad2deg(dd0.el), 1))

59 epochs collected
epoch 0 common sats: [3, 4, 6, 9, 14, 17, 19, 22, 28, 35, 39, 40, 45, 47, 53, 58]
epoch 0 elevations [deg]: [40.8 35.7 40.9 33.  25.2 85.4 61.6 16.  32.1 32.8 17.9 48.6 60.9 41.4
 27.8 18.7]


## 3. Reference satellite per constellation (gauge)

Only between-satellite double differences are observable, so one single-difference
ambiguity per constellation is unobservable. We pick the highest-elevation
satellite as the reference and later pin its ambiguity (any value works -- the DDs
are gauge-independent).

In [4]:
el_cum = {}
for (_, _, dd) in frames:
    for k, s in enumerate(dd.sat):
        if dd.el[k] > 0:
            el_cum[int(s)] = el_cum.get(int(s), 0.0) + dd.el[k]
ref_of = {}
for s, e in el_cum.items():
    sys = sat2prn(s)[0]
    if sys in SYSS and (sys not in ref_of or e > el_cum[ref_of[sys]]):
        ref_of[sys] = s
print('reference satellite per constellation:',
      {int(k): int(v) for k, v in ref_of.items()})

reference satellite per constellation: {0: 17, 1: 45}


## 4. Build the factor graph incrementally (ISAM2) and resolve each epoch

For every epoch and every frequency we add, per target satellite:
a `DoubleDifferencePseudorangeFactor` and a `DoubleDifferenceCarrierPhaseFactor`
(both form the rover-base double difference internally via Sagnac-corrected
`gnss::geodist`). The carrier factor ties the reference and target
**between-receiver SD ambiguities**; the reference ambiguity is gauge-pinned to its
carrier-minus-code value (non-zero so cssrlib's `ddidx` can pivot on it).

After each `isam.update` we attempt integer AR (next section).
ISAM2 uses **QR** factorization, whose joint marginals are robust.

In [5]:
params = gtsam.ISAM2Params(); params.setFactorization('QR')
isam = gtsam.ISAM2(params)
seen_am, pinned = set(), set()
first_fix, n_fix, nfac = None, 0, 0

for ei, (obs, obsb, dd) in enumerate(frames):
    graph = gtsam.NonlinearFactorGraph()
    val = gtsam.Values()
    if ei == 0:
        val.insert(X, gtsam.Point3(*dec.pos))
        graph.add(gtsam.PriorFactorPoint3(
            X, gtsam.Point3(*dec.pos), gtsam.noiseModel.Isotropic.Sigma(3, 30.0)))

    by_sys = {}
    for k, s in enumerate(dd.sat):
        by_sys.setdefault(sat2prn(int(s))[0], []).append(k)
    for sys, ks in by_sys.items():
        ref = ref_of.get(sys)
        ridx = next((k for k in ks if int(dd.sat[k]) == ref), None)
        if ridx is None:
            continue
        for f in range(nf):
            lam = obs.sig[sys][uTYP.L][f].wavelength()
            pr_rr, pr_br = obs.P[dd.iu[ridx], f], obsb.P[dd.ir[ridx], f]
            cp_rr = obs.L[dd.iu[ridx], f] * lam
            cp_br = obsb.L[dd.ir[ridx], f] * lam
            if 0.0 in (pr_rr, pr_br, cp_rr, cp_br):
                continue
            rs_ref, rsb_ref = dd.rs[dd.iu[ridx], :3], dd.rsb[dd.ir[ridx], :3]
            sd_ref = ((cp_rr - cp_br) - (pr_rr - pr_br)) / lam   # gauge value
            if AM(ref, f) not in seen_am:
                val.insert(AM(ref, f), float(sd_ref)); seen_am.add(AM(ref, f))
            if (ref, f) not in pinned:
                graph.addPriorDouble(AM(ref, f), sd_ref,
                                     gtsam.noiseModel.Isotropic.Sigma(1, 0.5))
                pinned.add((ref, f))
            for k in ks:
                js = int(dd.sat[k])
                if k == ridx:
                    continue
                pr_tr, pr_tb = obs.P[dd.iu[k], f], obsb.P[dd.ir[k], f]
                cp_tr = obs.L[dd.iu[k], f] * lam
                cp_tb = obsb.L[dd.ir[k], f] * lam
                if 0.0 in (pr_tr, pr_tb, cp_tr, cp_tb):
                    continue
                rs_j, rsb_j = dd.rs[dd.iu[k], :3], dd.rsb[dd.ir[k], :3]
                w = 1.0 / max(np.sin(min(dd.el[k], dd.el[ridx])), 0.1)
                graph.add(gtsam.DoubleDifferencePseudorangeFactor(
                    X, pr_rr, pr_br, pr_tr, pr_tb,
                    gtsam.Point3(*rs_ref), gtsam.Point3(*rs_j),
                    gtsam.Point3(*rsb_ref), gtsam.Point3(*rsb_j),
                    gtsam.Point3(*rb), gtsam.noiseModel.Isotropic.Sigma(1, 0.3 * w)))
                sd_tgt = ((cp_tr - cp_tb) - (pr_tr - pr_tb)) / lam
                if AM(js, f) not in seen_am:
                    val.insert(AM(js, f), float(sd_tgt)); seen_am.add(AM(js, f))
                graph.add(gtsam.DoubleDifferenceCarrierPhaseFactor(
                    X, AM(ref, f), AM(js, f), cp_rr, cp_br, cp_tr, cp_tb,
                    gtsam.Point3(*rs_ref), gtsam.Point3(*rs_j),
                    gtsam.Point3(*rsb_ref), gtsam.Point3(*rsb_j),
                    gtsam.Point3(*rb), lam,
                    gtsam.noiseModel.Isotropic.Sigma(1, 0.01 * w)))
    nfac += graph.size()
    isam.update(graph, val)

    # ---- integer AR on the current float (LAMBDA via the gnss_ar bridge) ----
    res = isam.calculateEstimate()
    nb, xa = resolve_ar(rtk, isam, res, X, AM, dd.sat, dd.el, seen_am, nf, SYSS)
    xh = xa if nb > 0 else np.array(res.atPoint3(X))
    if nb > 0:
        n_fix += 1
        first_fix = ei if first_fix is None else first_fix
    enu = gn.ecef2enu(pos_ref, xh - xyz_ref)
    if ei % 10 == 0 or ei == len(frames) - 1:
        print(f'ep{ei:3d} {"FIX " if nb > 0 else "flt "} nb={nb:2d} '
              f'2D={np.hypot(enu[0], enu[1]):.3f} 3D={np.linalg.norm(xh - xyz_ref):.3f} m')
print(f'\ngraph: {nfac} factors, {len(seen_am)} ambiguities')

ep  0 FIX  nb=28 2D=0.009 3D=0.018 m


ep 10 FIX  nb=28 2D=0.007 3D=0.014 m


ep 20 FIX  nb=28 2D=0.007 3D=0.013 m


ep 30 FIX  nb=28 2D=0.007 3D=0.013 m


ep 40 FIX  nb=30 2D=0.007 3D=0.015 m


ep 50 FIX  nb=30 2D=0.008 3D=0.016 m


ep 58 FIX  nb=28 2D=0.007 3D=0.015 m

graph: 3393 factors, 34 ambiguities


## 5. Integer ambiguity resolution -- the AR bridge

`resolve_ar` (in `gnss_ar.py`) writes the GTSAM float ambiguities and their joint
covariance into the cssrlib `nav` state, then calls `resamb_lambda` (LAMBDA + ddidx
single-difference + ratio test). The **full position+ambiguity joint marginal is
ill-conditioned (NaN)**, so the covariance is assembled from the *ambiguity-only*
joint plus *pairwise* (position, ambiguity) cross terms -- numerically stable.
This is the AR algorithm only; cssrlib's EKF is not used.

## 6. Results

With ~23 satellites on two frequencies and a short baseline, RTK fixes
**instantaneously (epoch 0)** and stays fixed, agreeing with the surveyed marker
at the **mm-cm** level.

In [6]:
print(f'first fix: epoch {first_fix}   fixed {n_fix}/{len(frames)} epochs')

first fix: epoch 0   fixed 59/59 epochs
